In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
import os
import matplotlib.pyplot as plt
import blink_features

def get_head_movement_pattern(duration=20, visualize=True):
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose()
    cap = cv2.VideoCapture(0)
    print("Look at the camera and move your head. ESC to quit early.")
    nose_x = []
    nose_y = []
    timestamps = []
    h, w = None, None
    start_time = time.time()
    frame_count = 0
    # -------- BLINK INITIALIZATION --------
    blink_state = blink_features.init_blink_state()
    # --------------------------------------
    while time.time() - start_time < duration:
        ret, frame = cap.read()
        if not ret:
            break
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results_pose = pose.process(rgb_frame)
        if h is None or w is None:
            h, w, _ = frame.shape
        if results_pose.pose_landmarks:
            landmarks = results_pose.pose_landmarks.landmark
            nose = landmarks[0]
            nose_x.append(nose.x)
            nose_y.append(nose.y)
            timestamps.append(time.time() - start_time)
            if visualize:
                cx, cy = int(nose.x * w), int(nose.y * h)
                cv2.circle(frame, (cx, cy), 5, (255, 0, 0), -1)
        frame_count += 1
        # ------ BLINK PROCESSING PER FRAME -------
        blink_state = blink_features.process_blink_frame(frame, blink_state, visualize=visualize)
        # ----------------------------------------
        if visualize:
            cv2.putText(
                frame,
                f"Time: {int(duration - (time.time() - start_time))}s",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2
            )
            cv2.putText(
                frame,
                f"Frames: {len(nose_x)}",
                (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
            )
            cv2.imshow("Head Movement Pattern Capture", frame)
            if cv2.waitKey(1) & 0xFF == 27:
                print("Stopped by user.")
                break
    cap.release()
    cv2.destroyAllWindows()
    
    # ------- GET AND PRINT BLINK RESULTS -------
    blink_stats = blink_features.get_blink_stats(blink_state)
    print(f"Blinks counted: {blink_stats['blink_count']}")
    print(f"Avg blink duration: {blink_stats['avg_blink_duration']:.2f} seconds")
    # ------------------------------------------
    # Return comprehensive pattern data INCLUDING BLINK INFO
    pattern_data = {
        'nose_x': np.array(nose_x),
        'nose_y': np.array(nose_y),
        'timestamps': np.array(timestamps),
        'duration': duration,
        'frame_count': len(nose_x),
        'blink_count': blink_stats['blink_count'],
        'avg_blink_duration': blink_stats['avg_blink_duration']
    }
    return pattern_data

# ENHANCED: Multi-pattern support functions
def save_pattern(pattern_data, user_id, pattern_number=1):
    """Save a numbered pattern for a user"""
    filename = f"{user_id}_head_pattern_{pattern_number}.npy"
    np.save(filename, pattern_data)
    print(f"Pattern {pattern_number} saved for {user_id} as {filename}")
    print(f"Captured {pattern_data['frame_count']} frames over {pattern_data['duration']} seconds")
    print(f"Stored {pattern_data['blink_count']} blinks")
    return filename

def load_all_patterns(user_id):
    """Load all stored patterns for a user"""
    patterns = []
    pattern_number = 1
    
    while True:
        filename = f"{user_id}_head_pattern_{pattern_number}.npy"
        if os.path.exists(filename):
            patterns.append(np.load(filename, allow_pickle=True).item())
            pattern_number += 1
        else:
            break
    
    return patterns

def load_pattern(filename):
    return np.load(filename, allow_pickle=True).item()

def interpolate_pattern(pattern, target_length):
    """Interpolate pattern to match target length for comparison"""
    if len(pattern) == target_length:
        return pattern
    
    original_indices = np.linspace(0, len(pattern) - 1, len(pattern))
    target_indices = np.linspace(0, len(pattern) - 1, target_length)
    interpolated = np.interp(target_indices, original_indices, pattern)
    return interpolated

def calculate_pattern_features(nose_x, nose_y):
    """Extract meaningful features from movement patterns"""
    features = {}
    
    # Movement range
    features['x_range'] = np.max(nose_x) - np.min(nose_x)
    features['y_range'] = np.max(nose_y) - np.min(nose_y)
    
    # Movement variance
    features['x_variance'] = np.var(nose_x)
    features['y_variance'] = np.var(nose_y)
    
    # Movement speed (derivatives)
    x_speed = np.abs(np.diff(nose_x))
    y_speed = np.abs(np.diff(nose_y))
    features['avg_x_speed'] = np.mean(x_speed)
    features['avg_y_speed'] = np.mean(y_speed)
    
    # Movement frequency analysis
    x_fft = np.fft.fft(nose_x)
    dominant_freq_x = np.argmax(np.abs(x_fft[1:len(x_fft)//2])) + 1
    features['dominant_freq_x'] = dominant_freq_x
    
    # Pattern smoothness
    x_smoothness = np.mean(np.abs(np.diff(nose_x, n=2)))  # Second derivative
    y_smoothness = np.mean(np.abs(np.diff(nose_y, n=2)))
    features['x_smoothness'] = x_smoothness
    features['y_smoothness'] = y_smoothness
    
    return features

def compare_patterns(ref_pattern_data, test_pattern_data):
    """Enhanced pattern comparison with head movement AND blink count"""
    
    ref_x = ref_pattern_data['nose_x']
    ref_y = ref_pattern_data['nose_y']
    test_x = test_pattern_data['nose_x']
    test_y = test_pattern_data['nose_y']
    
    # Ensure we have enough data
    if len(ref_x) < 10 or len(test_x) < 10:
        print("Warning: Not enough movement data captured")
        return 1.0  # High difference = fail authentication
    
    # Interpolate to same length for direct comparison
    target_length = min(len(ref_x), len(test_x), 100)  # Limit to reasonable size
    ref_x_interp = interpolate_pattern(ref_x, target_length)
    ref_y_interp = interpolate_pattern(ref_y, target_length)
    test_x_interp = interpolate_pattern(test_x, target_length)
    test_y_interp = interpolate_pattern(test_y, target_length)
    
    # Calculate feature-based comparison
    ref_features = calculate_pattern_features(ref_x, ref_y)
    test_features = calculate_pattern_features(test_x, test_y)
    
    # Normalize patterns for comparison
    ref_x_norm = (ref_x_interp - np.mean(ref_x_interp)) / (np.std(ref_x_interp) + 1e-8)
    test_x_norm = (test_x_interp - np.mean(test_x_interp)) / (np.std(test_x_interp) + 1e-8)
    ref_y_norm = (ref_y_interp - np.mean(ref_y_interp)) / (np.std(ref_y_interp) + 1e-8)
    test_y_norm = (test_y_interp - np.mean(test_y_interp)) / (np.std(test_y_interp) + 1e-8)
    
    # Multiple comparison metrics
    metrics = {}
    
    # 1. Direct pattern similarity (normalized)
    x_diff = np.mean(np.abs(ref_x_norm - test_x_norm))
    y_diff = np.mean(np.abs(ref_y_norm - test_y_norm))
    metrics['pattern_diff'] = (x_diff + y_diff) / 2
    
    # 2. Correlation coefficient
    x_corr = np.corrcoef(ref_x_norm, test_x_norm)[0, 1]
    y_corr = np.corrcoef(ref_y_norm, test_y_norm)[0, 1]
    x_corr = 0 if np.isnan(x_corr) else x_corr
    y_corr = 0 if np.isnan(y_corr) else y_corr
    metrics['correlation'] = (x_corr + y_corr) / 2
    
    # 3. Feature similarity
    feature_diffs = []
    for key in ref_features.keys():
        if ref_features[key] != 0:
            diff = abs(ref_features[key] - test_features[key]) / abs(ref_features[key])
            feature_diffs.append(min(diff, 2.0))  # Cap at 200% difference
        else:
            feature_diffs.append(1.0 if test_features[key] != 0 else 0.0)
    
    metrics['feature_diff'] = np.mean(feature_diffs)
    
    # ===== NEW: BLINK COUNT COMPARISON =====
    ref_blink_count = ref_pattern_data.get('blink_count', 0)
    test_blink_count = test_pattern_data.get('blink_count', 0)
    ref_avg_duration = ref_pattern_data.get('avg_blink_duration', 0)
    test_avg_duration = test_pattern_data.get('avg_blink_duration', 0)
    
    # Blink count similarity (normalized by time duration)
    ref_blink_rate = ref_blink_count / ref_pattern_data['duration']
    test_blink_rate = test_blink_count / test_pattern_data['duration']
    
    blink_rate_diff = abs(ref_blink_rate - test_blink_rate) / (ref_blink_rate + 1e-8)
    blink_rate_diff = min(blink_rate_diff, 2.0)  # Cap at 200%
    
    # Blink duration similarity
    if ref_avg_duration > 0 and test_avg_duration > 0:
        duration_diff = abs(ref_avg_duration - test_avg_duration) / ref_avg_duration
        duration_diff = min(duration_diff, 2.0)
    else:
        duration_diff = 1.0 if (ref_avg_duration == 0) != (test_avg_duration == 0) else 0.0
    
    # Combined blink difference
    metrics['blink_diff'] = (blink_rate_diff * 0.7 + duration_diff * 0.3)
    
    # Enhanced combined score including blink data
    correlation_diff = 1 - max(metrics['correlation'], 0)
    
    combined_diff = (metrics['pattern_diff'] * 0.35 + 
                    correlation_diff * 0.25 + 
                    metrics['feature_diff'] * 0.25 +
                    metrics['blink_diff'] * 0.15)  # 15% weight for blink patterns
    
    print(f"\n--- Enhanced Pattern Analysis ---")
    print(f"Reference pattern: {len(ref_x)} frames")
    print(f"Test pattern: {len(test_x)} frames")
    print(f"Pattern difference: {metrics['pattern_diff']:.4f}")
    print(f"Correlation: {metrics['correlation']:.4f}")
    print(f"Feature difference: {metrics['feature_diff']:.4f}")
    
    # NEW: Blink analysis output
    print(f"\n--- Blink Analysis ---")
    print(f"Reference blinks: {ref_blink_count} ({ref_blink_rate:.2f}/sec)")
    print(f"Test blinks: {test_blink_count} ({test_blink_rate:.2f}/sec)")
    print(f"Blink rate difference: {blink_rate_diff:.4f}")
    print(f"Avg duration - Ref: {ref_avg_duration:.3f}s, Test: {test_avg_duration:.3f}s")
    print(f"Duration difference: {duration_diff:.4f}")
    print(f"Combined blink difference: {metrics['blink_diff']:.4f}")
    
    print(f"\n--- Final Score ---")
    print(f"Combined difference: {combined_diff:.4f}")
    
    return combined_diff

def authenticate_against_multiple_patterns(test_pattern_data, user_id):
    """Authenticate by comparing against all stored patterns"""
    stored_patterns = load_all_patterns(user_id)
    
    if not stored_patterns:
        print(" No stored patterns found. Please register first!")
        return False, 1.0, 0, 0.6
    
    print(f" Found {len(stored_patterns)} stored patterns for {user_id}")
    
    best_match_score = float('inf')
    best_pattern_number = 0
    all_scores = []
    
    for i, ref_pattern in enumerate(stored_patterns, 1):
        print(f"\n--- Comparing against Pattern {i} ---")
        diff = compare_patterns(ref_pattern, test_pattern_data)
        all_scores.append(diff)
        
        if diff < best_match_score:
            best_match_score = diff
            best_pattern_number = i
        
        print(f"Pattern {i} score: {diff:.4f}")
    
    # Use adaptive threshold based on pattern complexity                #############################
    base_threshold = 0.8
    ref_complexity = np.std(stored_patterns[best_pattern_number-1]['nose_x']) + \
                    np.std(stored_patterns[best_pattern_number-1]['nose_y'])
    
    threshold = base_threshold + (0.2 if ref_complexity > 0.1 else 0.0)
    
    print(f"\n--- Multi-Pattern Authentication Results ---")
    print(f" Best match: Pattern {best_pattern_number} (score: {best_match_score:.4f})")
    print(f" All pattern scores: {[f'{score:.4f}' for score in all_scores]}")
    print(f"🎚 Threshold: {threshold:.3f}")
    
    is_authenticated = best_match_score < threshold
    return is_authenticated, best_match_score, best_pattern_number, threshold

if __name__ == "__main__":
    print("= Enhanced Head Movement + Blink Authentication System =")
    print("Choose an option:")
    print("1 - Register head movement pattern(s)")
    print("2 - Authenticate by matching head movement + blink patterns")
    choice = input("Enter 1 or 2: ").strip()
    user_id = input("Enter your user ID or name: ").strip()
    
    if choice == "1":
        # Check existing patterns
        existing_patterns = load_all_patterns(user_id)
        if existing_patterns:
            print(f"\n Found {len(existing_patterns)} existing patterns for {user_id}")
            print("1 - Add more patterns")
            print("2 - Replace all patterns")
            sub_choice = input("Enter 1 or 2: ").strip()
            
            if sub_choice == "2":
                # Delete existing patterns
                pattern_num = 1
                while True:
                    filename = f"{user_id}_head_pattern_{pattern_num}.npy"
                    if os.path.exists(filename):
                        os.remove(filename)
                        pattern_num += 1
                    else:
                        break
                print(" Previous patterns deleted")
                start_pattern_num = 1
            else:
                start_pattern_num = len(existing_patterns) + 1
        else:
            start_pattern_num = 1
        
        # Get number of patterns to record
        max_patterns = 5
        patterns_to_record = int(input(f"How many patterns would you like to register? (1-{max_patterns}): "))
        patterns_to_record = max(1, min(max_patterns, patterns_to_record))
        
        successful_patterns = 0
        
        for pattern_num in range(start_pattern_num, start_pattern_num + patterns_to_record):
            print(f"\n--- Recording Pattern {pattern_num} ---")
            print("- Move your head left/right naturally")
            print("- Blink naturally during recording")
            print("- Try different movements: slow, fast, circular, nodding")
            print("- Make sure you move significantly for 20 seconds")
            input(f"Press Enter when ready for pattern {pattern_num}...")
            
            ref_pattern_data = get_head_movement_pattern(duration=20, visualize=True)
            
            if ref_pattern_data['frame_count'] < 10:
                print(" Not enough movement detected. Skipping this pattern.")
                continue
            else:
                save_pattern(ref_pattern_data, user_id, pattern_num)
                successful_patterns += 1
                
                # EXACT SAME PLOT AS YOUR ORIGINAL - Plot registration pattern
                plt.figure(figsize=(12, 8))
                
                plt.subplot(2, 2, 1)
                plt.plot(ref_pattern_data['nose_x'])
                plt.xlabel("Frame")
                plt.ylabel("Nose X Position")
                plt.title("Head Movement X (Registered)")
                
                plt.subplot(2, 2, 2)
                plt.plot(ref_pattern_data['nose_y'])
                plt.xlabel("Frame")
                plt.ylabel("Nose Y Position")
                plt.title("Head Movement Y (Registered)")
                
                plt.subplot(2, 2, 3)
                plt.plot(ref_pattern_data['timestamps'], ref_pattern_data['nose_x'])
                plt.xlabel("Time (seconds)")
                plt.ylabel("Nose X Position")
                plt.title("X Movement vs Time")
                
                plt.subplot(2, 2, 4)
                plt.plot(ref_pattern_data['timestamps'], ref_pattern_data['nose_y'])
                plt.xlabel("Time (seconds)")
                plt.ylabel("Nose Y Position")
                plt.title("Y Movement vs Time")
                
                plt.tight_layout()
                plt.show()
                
                print(f" Pattern {pattern_num} saved successfully!")
        
        print(f"\n Registration completed for {user_id}")
        print(f" Total patterns stored: {len(load_all_patterns(user_id))}")
        
    elif choice == "2":
        stored_patterns = load_all_patterns(user_id)
        if not stored_patterns:
            print(" No saved patterns found. Please register first!")
        else:
            print(f"\n Authentication Mode for {user_id}")
            print(f" Found {len(stored_patterns)} registered patterns")
            print("- Move your head naturally")
            print("- Blink naturally during authentication")
            print("- System will match head movement AND blink patterns")
            input("Press Enter when ready...")
            
            test_pattern_data = get_head_movement_pattern(duration=20, visualize=True)
            
            if test_pattern_data['frame_count'] < 10:
                print(" Not enough movement detected. Authentication failed.")
            else:
                is_authenticated, best_score, best_pattern, threshold = \
                    authenticate_against_multiple_patterns(test_pattern_data, user_id)
                
                # Get the best matching reference pattern for plotting
                ref_pattern_data = stored_patterns[best_pattern - 1]
                diff = best_score
                
                # EXACT SAME PLOT AS YOUR ORIGINAL - Plot comparison
                plt.figure(figsize=(15, 10))
                
                plt.subplot(2, 3, 1)
                plt.plot(ref_pattern_data['nose_x'], label="Registered", linewidth=2)
                plt.plot(test_pattern_data['nose_x'], label="Current", alpha=0.7)
                plt.xlabel("Frame")
                plt.ylabel("Nose X Position")
                plt.title("X Movement Comparison")
                plt.legend()
                
                plt.subplot(2, 3, 2)
                plt.plot(ref_pattern_data['nose_y'], label="Registered", linewidth=2)
                plt.plot(test_pattern_data['nose_y'], label="Current", alpha=0.7)
                plt.xlabel("Frame")
                plt.ylabel("Nose Y Position")
                plt.title("Y Movement Comparison")
                plt.legend()
                
                plt.subplot(2, 3, 3)
                plt.scatter(ref_pattern_data['nose_x'], ref_pattern_data['nose_y'], 
                           alpha=0.6, label="Registered", s=20)
                plt.scatter(test_pattern_data['nose_x'], test_pattern_data['nose_y'], 
                           alpha=0.6, label="Current", s=20)
                plt.xlabel("X Position")
                plt.ylabel("Y Position")
                plt.title("Movement Path")
                plt.legend()
                
                # Speed analysis
                plt.subplot(2, 3, 4)
                ref_speed = np.sqrt(np.diff(ref_pattern_data['nose_x'])**2 + 
                                   np.diff(ref_pattern_data['nose_y'])**2)
                test_speed = np.sqrt(np.diff(test_pattern_data['nose_x'])**2 + 
                                    np.diff(test_pattern_data['nose_y'])**2)
                plt.plot(ref_speed, label="Registered Speed", linewidth=2)
                plt.plot(test_speed, label="Current Speed", alpha=0.7)
                plt.xlabel("Frame")
                plt.ylabel("Movement Speed")
                plt.title("Movement Speed Comparison")
                plt.legend()
                
                plt.subplot(2, 3, 5)
                plt.hist(ref_pattern_data['nose_x'], alpha=0.7, bins=20, label="Registered X")
                plt.hist(test_pattern_data['nose_x'], alpha=0.7, bins=20, label="Current X")
                plt.xlabel("X Position")
                plt.ylabel("Frequency")
                plt.title("X Position Distribution")
                plt.legend()
                
                plt.subplot(2, 3, 6)
                plt.text(0.1, 0.8, f"Combined Difference: {diff:.4f}", transform=plt.gca().transAxes, fontsize=12)
                plt.text(0.1, 0.6, f"Threshold: {threshold:.3f}", transform=plt.gca().transAxes, fontsize=12)
                plt.text(0.1, 0.4, f"Result: {' MATCH' if diff < threshold else ' NO MATCH'}", 
                        transform=plt.gca().transAxes, fontsize=14, 
                        color='green' if diff < threshold else 'red')
                plt.text(0.1, 0.2, f"Frames: {test_pattern_data['frame_count']}", transform=plt.gca().transAxes, fontsize=10)
                plt.axis('off')
                plt.title("Authentication Result")
                
                plt.tight_layout()
                plt.show()
                
                if is_authenticated:
                    print(f"\n Authentication SUCCESSFUL!")
                    print(f" Matched against Pattern {best_pattern}")
                    print(f" Confidence: {(1-best_score)*100:.1f}%")
                    print(f" Welcome, {user_id}!")
                else:
                    print(f"\n Authentication FAILED")
                    print(f" Best match score: {best_score:.4f}")
                    print(f" Required threshold: {threshold:.3f}")
                    print(f" Gap: {best_score - threshold:.4f}")
                    print(" Try moving more naturally or register additional patterns")
    else:
        print("Invalid choice.")


In [ ]:
print(mp.version)